In [0]:
df = spark.read.parquet("abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_201_99457.parquet").select("customer_id","category","quantity","invoice_date")

In [0]:
display(df.limit(5))

25409792
20000000

In [0]:
df_union = df
expected_rows = 20000000

while df_union.count() <= expected_rows:
    df_union = df_union.union(df_union)
    print(f"count: {df_union.count()}")

print(f"Final Count: {df_union.count()}")

In [0]:
df_union.write.mode("overwrite").saveAsTable("deltacatalog.deltadb.zorder_ex1")

In [0]:
%%time
spark.sql(
    """
        Select category,
            SUM(price*quantity) as total_sales
        FROM deltacatalog.deltadb.zorder_ex1
            where customer_id=201
            groupBy category
    """
).show()

In [0]:
%sql
Optimize deltacatalog.deltadb.zorder_ex1
ZORDER BY customer_id

In [0]:
# Or use this Pyspark Version
# from delta.tables import DeltaTable

# table = DeltaTable.forName(spark,"deltacatalog.deltadb.zorder_ex1")
# table.optimize().executeZOrderBy("customer_id")

In [0]:
%%time
spark.sql(
    """
        Select category,
            SUM(price*quantity) as total_sales
        FROM deltacatalog.deltadb.zorder_ex1
            where customer_id=201
            groupBy category
    """
).show()

Example 2

In [0]:
df.write.mode("overwrite").partitionBy("invoice_date").saveAsTable("deltacatalog.deltadb.zorder_ex2")

- Hive Style Partition : `invoice_date`
- ZORDER BY : `customer_id`
- For each of those `invoice_date`

In [0]:
from pyspark.sql.functions import *

In [0]:
df_new_partition = df.filter(col("invoice_date")=='2023-01-01').withColumn("invoice_date",to_date(lit("2026-05-22")))

In [0]:
df_new_partition.write.mode("append").partitionBy("invoice_date").saveAsTable("deltacatalog.deltadb.zorder_ex2")

In [0]:
%sql
Select Max(invoice_date) from deltacatalog.deltadb.zorder_ex2

In [0]:
%sql
OPTIMIZE deltacatalog.deltadb.zorder_ex2
where invoice_date = '2026-05-22'
ZORDER BY customer_id